# Checkpoint E — Experimental population and coverage (E35–E43)

**Scope:** recorded-host population, first-exposure timestamps, activity and answerability. Technical description only: **no ITT claim, KPI, causal effects or Control–Test superiority**. All 14,708 registered hosts, including zero-project hosts, remain in denominator descriptions. Checkpoint C's UTC time conventions and incomplete-last-day caveat apply.

**Reproduction:** Python 3.13, raw CSVs in `data/raw/`, run all notebook cells from a clean Jupyter kernel. No notebook output can substitute for independent local V7 validation.

In [1]:
from pathlib import Path
import sys
import pandas as pd
import numpy as np
import plotly.express as px
from IPython.display import display
here=Path.cwd().resolve()
ROOT=next((v for v in [here,*here.parents] if (v/'pyproject.toml').is_file() and (v/'data/raw').is_dir()),None)
if ROOT is None:raise FileNotFoundError('Repository root with pyproject.toml and data/raw/ required')
if str(ROOT) not in sys.path:sys.path.insert(0,str(ROOT))
from src.foundation_a import load_raw,assert_conserved
from src.foundation_ce import utc_timestamp as ts, parse_utc, host_experiment_counts
f,provenance=load_raw(ROOT)
e,a,p=f['exposures'],f['assignments'],f['payments']
assert (len(e),len(a),len(p)) == (14708,279847,306957), 'Source differs from checkpoints A/B: review before continuing'
assert not e.host_id.isna().any() and not e.host_id.duplicated().any()
print('Python',sys.version.split()[0]);display(provenance)
EX0,EXEND,BASE=ts('2025-08-18'),ts('2025-11-20'),ts('2025-07-18')
h=host_experiment_counts(e,a)
h['exposure_utc']=parse_utc(h.first_exposed_at)
a['completed_utc']=parse_utc(a.completed_at)
assert_conserved(len(e),h.shape[0])
assert (h.exposure_utc.notna()).all()

Python 3.13.5


,dataset,file,bytes,sha256,columns_unexpected
0,exposures,experiment_exposures.csv,481724,9cce67cd8d90d292c0703416b3c28c69783c56d10c2f99...,
1,assignments,assignments.csv,16411731,c66a6d5cb89b62bc3542ee9184e75c8ded41589e91dac9...,
2,payments,payment_logs.csv,24857672,ab6bf97f2ca216504058b5de1deb2f80641d783e8e2365...,


## E35 — Registered population by group and recorded first exposure
**Procedure:** count registered hosts by group, projects from all completed assignments, exposure cohorts by UTC month. **Validation:** groups reconcile to the registered-host universe; projects to the assignment universe. **Limitation:** exposed/registered hosts in this extract do not necessarily equal the original randomized population. **Status: answered for documented records.**

In [2]:
gr=h.groupby('experiment_group').agg(registered_hosts=('host_id','size'),hosts_with_experiment_completions=('experiment_completions',lambda s:int(s.gt(0).sum())),experiment_projects=('experiment_completions','sum')).reset_index()
all_project_counts=a.merge(e[['host_id','experiment_group']],on='host_id',how='left',validate='many_to_one').groupby('experiment_group').size().rename('all_extract_projects')
gr['all_extract_projects']=gr.experiment_group.map(all_project_counts)
assert_conserved(len(e),int(gr.registered_hosts.sum()))
assert_conserved(len(a),int(gr.all_extract_projects.sum()))
display(gr)
h['exposure_month']=h.exposure_utc.dt.strftime('%Y-%m')
coh=h.groupby(['experiment_group','exposure_month']).agg(hosts=('host_id','size'),experiment_projects=('experiment_completions','sum')).reset_index()
assert_conserved(len(h),int(coh.hosts.sum()))
display(coh)
print('Observed first-exposure UTC bounds:',h.exposure_utc.min(),h.exposure_utc.max())

,experiment_group,registered_hosts,hosts_with_experiment_completions,experiment_projects,all_extract_projects
0,control,7377,6938,98855,137586
1,test,7331,6934,102447,142261


,experiment_group,exposure_month,hosts,experiment_projects
0,control,2025-08,4941,88909
1,control,2025-09,1309,7320
2,control,2025-10,818,2220
3,control,2025-11,309,406
4,test,2025-08,4845,92435
5,test,2025-09,1335,7070
6,test,2025-10,831,2499
7,test,2025-11,320,443


Observed first-exposure UTC bounds: 2025-08-18 13:15:07+00:00 2025-11-19 14:45:45+00:00


## E36 — Zero, one, or multiple completed projects per registered host
**Procedure:** count only projects completed during the documented experiment (`2025-08-18` inclusive to `2025-11-20` exclusive); compare full-extract project counts separately. Start from **exposures**, not assignments. **Validation:** exactly one bucket per registered host and project counts sum to the experiment project universe. **Status: answered for registered hosts and observed completions only.**

In [3]:
all_counts=a.groupby('host_id').size()
h['all_extract_completions']=h.host_id.map(all_counts).fillna(0).astype(int)
h['experiment_bucket']=np.select([h.experiment_completions.eq(0),h.experiment_completions.eq(1),h.experiment_completions.ge(2)],['zero','one','multiple'],default='unknown')
bybucket=h.groupby(['experiment_group','experiment_bucket']).agg(hosts=('host_id','size'),completed_experiment_projects=('experiment_completions','sum'),completed_all_extract_projects=('all_extract_completions','sum')).reset_index()
assert_conserved(len(h),int(bybucket.hosts.sum()))
assert_conserved(int(h.experiment_completions.sum()),int(bybucket.completed_experiment_projects.sum()))
assert_conserved(len(a),int(h.all_extract_completions.sum()))
display(bybucket)
print('Full extract zero-project hosts:',int(h.all_extract_completions.eq(0).sum()),'| Experiment window zero-completion hosts:',int(h.experiment_completions.eq(0).sum()))

,experiment_group,experiment_bucket,hosts,completed_experiment_projects,completed_all_extract_projects
0,control,multiple,5957,97874,135627
1,control,one,981,981,1738
2,control,zero,439,0,221
3,test,multiple,5991,101504,140473
4,test,one,943,943,1605
5,test,zero,397,0,183


Full extract zero-project hosts: 642 | Experiment window zero-completion hosts: 836


## E37 — Historical-first flags and observed recurrence
**Procedure:** audit positive `is_first_completed_project` flags and first observed completion chronologically; describe hosts with 2+ observed completions without naming this retention or loyalty. **Validation:** group totals, repeated flag checks, and observed minimum UTC; ties shown. **Limitation:** first completion ≠ registration date; history before 2025-07-18 is absent. **Status: partially answerable historically.**

In [4]:
flags=a.is_first_completed_project.eq('true')
flag_host_count=int(a.loc[flags,'host_id'].nunique())
flags_per_host=a.loc[flags].groupby('host_id').size()
per_host_earliest=a.groupby('host_id').completed_utc.min()
earliest=a.host_id.map(per_host_earliest)
flagged_later=flags & a.completed_utc.gt(earliest)
flagged_ties=a.loc[flags & a.completed_utc.eq(earliest)].groupby('host_id').size()
historical=pd.DataFrame([{'label':'positive_first_flags','value':int(flags.sum())},{'label':'hosts_with_positive_flag','value':flag_host_count},{'label':'hosts_with_multiple_positive_flags','value':int(flags_per_host.gt(1).sum())},{'label':'flags_later_than_observed_first_completion','value':int(flagged_later.sum())},{'label':'hosts_with_two_or_more_full_extract_completions','value':int(h.all_extract_completions.ge(2).sum())},{'label':'hosts_with_two_or_more_experiment_completions','value':int(h.experiment_completions.ge(2).sum())}]);display(historical)
assert int(flags.sum())==int(flags_per_host.sum())
assert int(h.all_extract_completions.sum())==len(a)
print('Observed earliest completion ties (hosts):',int(a.loc[a.completed_utc.eq(earliest)].groupby('host_id').size().gt(1).sum()))

,label,value
0,positive_first_flags,2755
1,hosts_with_positive_flag,2755
2,hosts_with_multiple_positive_flags,0
3,flags_later_than_observed_first_completion,0
4,hosts_with_two_or_more_full_extract_completions,12649
5,hosts_with_two_or_more_experiment_completions,11948


Observed earliest completion ties (hosts): 7


## E38 — Registered hosts represented in baseline and experiment
**Procedure:** create an exhaustive 2×2 presence matrix using documented UTC completion windows; preserve hosts with no projects. **Validation:** four disjoint cells sum to 14,708. **Limitation:** absence from the single-month baseline is not evidence of a newly registered host. **Status: answered for observed coverage.**

In [5]:
base_counts=a.loc[a.completed_utc.ge(BASE)&a.completed_utc.lt(EX0)].groupby('host_id').size()
h['baseline_completions']=h.host_id.map(base_counts).fillna(0).astype(int)
h['baseline_presence']=np.where(h.baseline_completions.gt(0),'baseline_yes','baseline_no')
h['experiment_presence']=np.where(h.experiment_completions.gt(0),'experiment_yes','experiment_no')
presence=pd.crosstab(h.baseline_presence,h.experiment_presence,dropna=False)
display(presence);assert_conserved(len(h),int(presence.to_numpy().sum()))
assert_conserved(len(a),int(h.experiment_completions.sum()),int(h.baseline_completions.sum()))
display(h.groupby(['experiment_group','baseline_presence','experiment_presence']).size().rename('registered_hosts').reset_index())

experiment_presence,experiment_no,experiment_yes
baseline_presence,,
baseline_no,642,3782
baseline_yes,194,10090


,experiment_group,baseline_presence,experiment_presence,registered_hosts
0,control,baseline_no,experiment_no,338
1,control,baseline_no,experiment_yes,1839
2,control,baseline_yes,experiment_no,101
3,control,baseline_yes,experiment_yes,5099
4,test,baseline_no,experiment_no,304
5,test,baseline_no,experiment_yes,1943
6,test,baseline_yes,experiment_no,93
7,test,baseline_yes,experiment_yes,4991


## E39 — Potential recorded-exposure opportunity, not demand time
**Procedure:** calculate elapsed days from each recorded `first_exposed_at` to the experiment's documented **calendar-day end (Nov 20 00:00 UTC)**. Display descriptives, exposure histogram and boxplot by registered group; compare with project counts **without dividing by time**. **Validation:** all host records retained, nonnegative duration; project sums conserved. **Limitation:** potential observation time does not equal actual treatment exposure, cleaning demand, or randomly assigned follow-up. **Status: observable with limitations.**

In [6]:
h['potential_exposure_days']=(EXEND-h.exposure_utc).dt.total_seconds()/86400
print('Exposure durations negative / missing:',int(h.potential_exposure_days.lt(0).sum()),int(h.potential_exposure_days.isna().sum()))
assert h.potential_exposure_days.ge(0).all()
display(h.groupby('experiment_group').potential_exposure_days.describe(percentiles=[.1,.25,.5,.75,.9]).round(2))
by_cohort=h.groupby(['experiment_group','exposure_month']).agg(hosts=('host_id','size'),median_potential_days=('potential_exposure_days','median'),zero_completion_hosts=('experiment_completions',lambda s:int(s.eq(0).sum())),experiment_completions=('experiment_completions','sum')).reset_index()
assert_conserved(len(h),int(by_cohort.hosts.sum()))
display(by_cohort)
fig=px.histogram(h, x='potential_exposure_days',color='experiment_group',nbins=24,title='Potential recorded-exposure days by experimental group (descriptive only)')
fig.update_layout(height=360)
display(fig)
fig2=px.box(h, x='experiment_group', y='potential_exposure_days', title='Potential exposure opportunity by group (not normalized project frequency)',points=False)
fig2.update_layout(height=350)
display(fig2)

Exposure durations negative / missing: 0 0


,count,mean,std,min,10%,25%,50%,75%,90%,max
experiment_group,,,,,,,,,,
control,7377.0,75.58,22.72,0.38,36.65,69.66,87.07,90.58,92.53,93.45
test,7331.0,75.33,22.81,0.48,37.36,68.61,86.91,90.58,92.52,93.44


,experiment_group,exposure_month,hosts,median_potential_days,zero_completion_hosts,experiment_completions
0,control,2025-08,4941,89.186470,133,88909
1,control,2025-09,1309,67.936528,114,7320
2,control,2025-10,818,35.259959,133,2220
3,control,2025-11,309,10.642141,59,406
4,test,2025-08,4845,89.536065,112,92435
5,test,2025-09,1335,67.978229,104,7070
6,test,2025-10,831,37.369109,127,2499
7,test,2025-11,320,11.416458,54,443


## E40 — Time-cohort composition and observation asymmetry
**Procedure:** tabulate exposure-month composition by assignment group and observed completions, separately from potential observation days. **Validation:** no project counted in multiple registered-host cohorts, and host totals conserved. **Limitation:** descriptive differences reflect observed composition; no normalization, treatment-effect claim, or eligibility rule. **Status: answered descriptively.**

In [7]:
cohort_table=h.groupby(['experiment_group','exposure_month']).agg(hosts=('host_id','size'),zero_experiment_completions=('experiment_completions',lambda x:int((x==0).sum())),one_experiment_completion=('experiment_completions',lambda x:int((x==1).sum())),multiple_experiment_completions=('experiment_completions',lambda x:int((x>=2).sum())),completions=('experiment_completions','sum'),min_potential_days=('potential_exposure_days','min'),median_potential_days=('potential_exposure_days','median'),max_potential_days=('potential_exposure_days','max')).reset_index()
assert_conserved(len(h),int(cohort_table.hosts.sum()))
assert_conserved(int(h.experiment_completions.sum()),int(cohort_table.completions.sum()))
assert (cohort_table.hosts == cohort_table[['zero_experiment_completions','one_experiment_completion','multiple_experiment_completions']].sum(axis=1)).all()
display(cohort_table.round(2))
print('Different potential observation durations are descriptive; no per-day project frequency is defined.')

,experiment_group,exposure_month,hosts,zero_experiment_completions,one_experiment_completion,multiple_experiment_completions,completions,min_potential_days,median_potential_days,max_potential_days
0,control,2025-08,4941,133,285,4523,88909,80.04,89.19,93.45
1,control,2025-09,1309,114,278,917,7320,50.02,67.94,79.99
2,control,2025-10,818,133,258,427,2220,19.08,35.26,49.91
3,control,2025-11,309,59,160,90,406,0.38,10.64,18.65
4,test,2025-08,4845,112,265,4468,92435,80.01,89.54,93.44
5,test,2025-09,1335,104,251,980,7070,50.04,67.98,79.98
6,test,2025-10,831,127,250,454,2499,19.04,37.37,49.99
7,test,2025-11,320,54,177,89,443,0.48,11.42,18.82


Different potential observation durations are descriptive; no per-day project frequency is defined.


## E41 — Randomization, registration, exposure, and treatment are different populations/events
**Procedure:** map each concept to available source evidence, whether directly verified, and missing audit fields. **Validation:** inspect actual CSV schema and source documentation rather than assuming randomization logs or technical feature assignment. **Limitation:** `experiment_group` and `first_exposed_at` identify recorded assignments/exposures only; ITT denominator and randomization instant are not independently established. **Status: partially answerable.**

In [8]:
claims=[('original randomized hosts','not present','No randomization log, enrollment roster, or exclusions','not identifiable from extract'),('registered experiment hosts','experiment_exposures.host_id / experiment_group','14708 unique recorded hosts','directly observable'),('host first exposure recorded','experiment_exposures.first_exposed_at','timestamp UTC; semantics from brief only','directly observable as recorded field'),('actual effective treatment exposure','not measured','No feature-delivery instrumentation / confirmation logs','not identifiable directly'),('randomization instant','not measured','No assignment-created timestamp','not identifiable directly'),('treatment applied per completed project','not measured','No per-project authorization or feature switch audit','not identifiable directly')]
exp_matrix=pd.DataFrame(claims,columns=['concept','available_evidence','technical_limitation_or_record','observability']);display(exp_matrix)
print('Fields present:',{name:list(df.columns) for name,df in f.items()})

,concept,available_evidence,technical_limitation_or_record,observability
0,original randomized hosts,not present,"No randomization log, enrollment roster, or ex...",not identifiable from extract
1,registered experiment hosts,experiment_exposures.host_id / experiment_group,14708 unique recorded hosts,directly observable
2,host first exposure recorded,experiment_exposures.first_exposed_at,timestamp UTC; semantics from brief only,directly observable as recorded field
3,actual effective treatment exposure,not measured,No feature-delivery instrumentation / confirma...,not identifiable directly
4,randomization instant,not measured,No assignment-created timestamp,not identifiable directly
5,treatment applied per completed project,not measured,No per-project authorization or feature switch...,not identifiable directly


Fields present: {'exposures': ['host_id', 'experiment_group', 'first_exposed_at'], 'assignments': ['assignment_id', 'host_id', 'created_at', 'completed_at', 'is_first_completed_project', 'completed_utc'], 'payments': ['payment_log_id', 'assignment_id', 'host_id', 'payment_method', 'payment_status', 'is_manual_attempt', 'created_at', 'captured_at']}


## E42 — Internal and external representativeness limits
**Procedure:** inventory exactly which units appear in each provided extract and which non-observed populations would be needed for extrapolation. **Validation:** schema and count evidence from E35–E38. **Limitation:** neither treatment-specific host activity nor the external host population is documented here. **Status: partially answerable; external representativeness not identifiable.**

In [9]:
rep=pd.DataFrame([{'observed_population':'registered hosts in experiment_exposures','records':len(h),'excluded_or_unknown':'Originally randomized hosts not supplied, if any'}, {'observed_population':'completed marketplace projects for those hosts','records':len(a),'excluded_or_unknown':'Cancelled, rejected, never accepted, and incomplete projects'}, {'observed_population':'logged payment attempts for those projects','records':len(p),'excluded_or_unknown':'Non-logged collection, operational communications and monetary ledger'}, {'observed_population':'external Turno hosts / other marketplaces','records':pd.NA,'excluded_or_unknown':'Selection, market, product and demographic coverage not provided'}]);display(rep)
print('Group counts cannot demonstrate representative assignment coverage or generalizability.')

,observed_population,records,excluded_or_unknown
0,registered hosts in experiment_exposures,14708,"Originally randomized hosts not supplied, if any"
1,completed marketplace projects for those hosts,279847,"Cancelled, rejected, never accepted, and incom..."
2,logged payment attempts for those projects,306957,"Non-logged collection, operational communicati..."
3,external Turno hosts / other marketplaces,<NA>,"Selection, market, product and demographic cov..."


Group counts cannot demonstrate representative assignment coverage or generalizability.


## E43 — Preliminary Data Answerability Matrix (technical observability only)
**Procedure:** classify directly identifiable evidence, observable-but-limited evidence and unavailable concepts. **Validation:** tie each classification to explicit CSV fields, checks in A/B/C/E, and the additional data required. **Limitation:** this is not approval of future denominators, estimands, success definitions or product KPIs. **Status: technically answered as a preliminary matrix.**

In [10]:
questions=[
('Completed projects per registered host','assignments.host_id, completed_at; exposures.host_id','directly identifiable','registered hosts only; completed projects only','original randomized roster, cancellation/acceptance logs'),
('Hosts with historical first completion','is_first_completed_project','observable with limitations','partial extract; no signup date or complete history','host signup and full historical activity'),
('Hosts with second observed completion','assignment_id, host_id, completed_at','observable with limitations','order ties, incomplete history/future observation','complete event history and agreed follow-up'),
('Timestamped capture per completed project','payment_logs.captured_at, assignment_id, payment_status','observable with limitations','B12 discordance; multiple capture records; no amounts','payment ledger; reconciliation policy'),
('Projects with no observed capture','payment_logs.captured_at, assignment_id','observable with limitations','absence is not non-payment; C23 horizon and missing logs','known complete event tracking and recovery ledger'),
('Timestamped captures grouped by host','payment_logs.captured_at, host_id','directly identifiable','records not amounts or successful settlement','ledger and definitions for financial success'),
('Elapsed time from completion to timestamped capture','assignments.completed_at, payment_logs.captured_at','observable with limitations','precompletion events; no timestamp for status-only successes; censoring','precise extraction cutoff, financial event audit'),
('Recovery after failed payment','payment_status, created_at, captured_at','observable with limitations','failure event time and retries may be unknown','actual failure, retry and alert event timestamps'),
('Method used and potential treatment-affected segment','payment_method in attempts','observable with limitations','method switches; conditioning on post-treatment method','pre-assignment primary method or declared eligibility flag'),
('Project acceptance rate','not present','not directly identifiable','only completed projects in assignments','project invitations / offers / acceptance / cancellation records'),
('Churn and loyalty','completed project timestamps only','not directly identifiable','history, demand, time-at-risk not complete','full host lifecycle, demand and established definitions'),
('Full financial settlement and impact','not present','not directly identifiable','no amounts due, captures by amount, credits or support costs','payment ledger, amounts, refunds and CS costs'),
('Original ITT population / treatment delivery','experiment_group, first_exposed_at','not directly identifiable','registered exposure log != proven randomization roster','randomization assignments, delivery audit logs')]
answerability=pd.DataFrame(questions,columns=['concept','field_evidence','technical_observability','limitation','additional_data_needed'])
display(answerability)
assert len(answerability)>=12
print('Categories:',answerability.technical_observability.value_counts().to_dict())

,concept,field_evidence,technical_observability,limitation,additional_data_needed
0,Completed projects per registered host,"assignments.host_id, completed_at; exposures.h...",directly identifiable,registered hosts only; completed projects only,"original randomized roster, cancellation/accep..."
1,Hosts with historical first completion,is_first_completed_project,observable with limitations,partial extract; no signup date or complete hi...,host signup and full historical activity
2,Hosts with second observed completion,"assignment_id, host_id, completed_at",observable with limitations,"order ties, incomplete history/future observation",complete event history and agreed follow-up
3,Timestamped capture per completed project,"payment_logs.captured_at, assignment_id, payme...",observable with limitations,B12 discordance; multiple capture records; no ...,payment ledger; reconciliation policy
4,Projects with no observed capture,"payment_logs.captured_at, assignment_id",observable with limitations,absence is not non-payment; C23 horizon and mi...,known complete event tracking and recovery ledger
5,Timestamped captures grouped by host,"payment_logs.captured_at, host_id",directly identifiable,records not amounts or successful settlement,ledger and definitions for financial success
6,Elapsed time from completion to timestamped ca...,"assignments.completed_at, payment_logs.capture...",observable with limitations,precompletion events; no timestamp for status-...,"precise extraction cutoff, financial event audit"
7,Recovery after failed payment,"payment_status, created_at, captured_at",observable with limitations,failure event time and retries may be unknown,"actual failure, retry and alert event timestamps"
8,Method used and potential treatment-affected s...,payment_method in attempts,observable with limitations,method switches; conditioning on post-treatmen...,pre-assignment primary method or declared elig...
9,Project acceptance rate,not present,not directly identifiable,only completed projects in assignments,project invitations / offers / acceptance / ca...


Categories: {'observable with limitations': 7, 'not directly identifiable': 4, 'directly identifiable': 2}


## E — Technical checkpoint conclusion
The observed registered-host universe includes zero-completion hosts and shows different potential observation durations according to recorded first-exposure timestamps. Neither original randomization population nor actual treatment delivery is identifiable from these files. C23 last-day observation uncertainty and B12 status/capture discordance remain dependencies for future metric approval. **No inferred Control–Test effects, ITT claim, or approved KPI is presented.** V7 remains OPEN pending proven clean-kernel execution.